## A larger LP: long-only portfolio allocation over real S&P 500 data

The toy example (`simplex_v2.ipynb`) has 2 variables and 3 constraints. This one uses ~300 real assets and ~300 constraints, to see the dense tableau simplex under real load.

Our `optix.simplex.solve` only supports **maximize**, all-`<=` constraints, and implicit `x_i >= 0`. So the objective is real (maximize expected return), but the constraint set is the *simplified* version of a real portfolio problem:

- **Budget**: total weight `<= 1`
- **Per-asset cap**: no single name `<= 5%` (concentration limit)
- **Sector cap**: no GICS sector `<= 25%` (diversification limit)

That's a real, if simplified, long-only allocation problem — no shorting, no variance term (that's Markowitz QP, a good target for a future notebook once we have a QP solver), no minimum-exposure or turnover-vs-benchmark constraints (those need `>=`/`=`, i.e. a two-phase simplex, also a good future notebook).

Data comes from `optix.datasets.sp500` — real S&P 500 constituents and daily prices, cached under `data/` so every notebook in this repo can reuse the same pull instead of re-fetching from Yahoo/Wikipedia each time.

In [1]:
import time
from collections import defaultdict

import polars as pl

from optix import Var
from optix.simplex import solve, to_tableau
from optix.datasets import load_returns, sample_universe

### Data: sample 300 S&P 500 names and their daily returns

`sample_universe` draws a reproducible random subset of tickers + GICS sectors; `load_returns` pulls (and caches) daily returns for the full S&P 500 so this cell doesn't hit the network on repeat runs.

In [2]:
sample = sample_universe(n=300, seed=42)
returns = load_returns()

available = set(returns.columns)
sample = sample.filter(pl.col("ticker").is_in(available))

tickers = sample["ticker"].to_list()
sector_of = dict(zip(sample["ticker"], sample["sector"]))
mean_returns = {t: returns[t].mean() * 252 for t in tickers}  # annualized

print(f"assets: {len(tickers)}")
sample["sector"].value_counts().sort("count", descending=True)

assets: 292


sector,count
str,u32
"""Industrials""",47
"""Financials""",46
"""Information Technology""",44
"""Health Care""",33
"""Consumer Discretionary""",27
…,…
"""Utilities""",22
"""Consumer Staples""",17
"""Communication Services""",14


### Problem: one `Var` per asset, three families of `<=` constraints

`sum(...)` works here because `Expression` supports both `+` and its reflected `__radd__`, so a generator of many terms folds into one `Expression` the way you'd hope.

In [4]:
PER_ASSET_CAP = 0.05  # no single name over 5%
SECTOR_CAP = 0.25  # no GICS sector over 25%

weight = {t: Var(t) for t in tickers}

objective = sum(mean_returns[t] * weight[t] for t in tickers)

by_sector = defaultdict(list)
for t in tickers:
    by_sector[sector_of[t]].append(t)

constraints = [sum(weight[t] for t in tickers) <= 1.0]
constraints += [weight[t] <= PER_ASSET_CAP for t in tickers]
constraints += [sum(weight[t] for t in members) <= SECTOR_CAP for members in by_sector.values()]

print(f"variables: {len(tickers)}, constraints: {len(constraints)}")

variables: 292, constraints: 304


### Tableau size and solve

With ~300 variables and ~300 constraints (plus one slack per constraint), the dense tableau is on the order of 300 x 600 floats — trivial memory, but a good order of magnitude past the toy example to actually watch the pivot loop work.

In [5]:
tbl, variables = to_tableau(objective, constraints)
print(f"tableau shape: {tbl.shape}, {tbl.nbytes / 1024:.0f} KB")

t0 = time.time()
solution = solve(objective, constraints, debug=                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    False )
print(f"solve time: {time.time() - t0:.3f}s") 

tableau shape: (305, 597), 1423 KB
solve time: 0.112s


### Sanity-check the solution and look at the allocation

Worth actually re-checking budget/cap/sector feasibility here rather than trusting it blindly — this is a much bigger tableau than anything the solver's been run on before, and it's exactly the kind of scale where a subtle pivoting bug (degeneracy, a stale mask, an off-by-one in the ratio test) would first show up.

In [6]:
weights = {t: solution.get(weight[t], 0.0) for t in tickers}
total_weight = sum(weights.values())
expected_return = sum(mean_returns[t] * w for t, w in weights.items())

assert total_weight <= 1.0 + 1e-6
for t, w in weights.items():
    assert w <= PER_ASSET_CAP + 1e-6, (t, w)
for sector, members in by_sector.items():
    sector_weight = sum(weights[t] for t in members)
    assert sector_weight <= SECTOR_CAP + 1e-6, (sector, sector_weight)

print(f"total weight allocated: {total_weight:.4f}")
print(f"expected annual return: {expected_return:.2%}")
print("budget, per-asset, and sector constraints all satisfied")

total weight allocated: 1.0000
expected annual return: 81.99%
budget, per-asset, and sector constraints all satisfied


**Caveat on that ~100% number**: it's not a forecast, it's trailing mean return annualized and then LP-maximized, which is exactly the setup that concentrates into whatever had the best 2023–2025 run (PLTR, COIN, HOOD, AVGO — the era's biggest winners). A real allocator would use a shrunk/forward-looking return estimate, not the trailing mean; this notebook is about stressing the solver, not about a portfolio you'd actually run.

In [7]:
positions = pl.DataFrame({
    "ticker": tickers,
    "sector": [sector_of[t] for t in tickers],
    "weight": [weights[t] for t in tickers],
}).filter(pl.col("weight") > 1e-9).sort("weight", descending=True)

print(f"nonzero positions: {len(positions)} of {len(tickers)}")
positions

nonzero positions: 20 of 292


ticker,sector,weight
str,str,f64
"""DELL""","""Information Technology""",0.05
"""FIX""","""Industrials""",0.05
"""KKR""","""Financials""",0.05
"""NRG""","""Utilities""",0.05
"""ARES""","""Financials""",0.05
…,…,…
"""HWM""","""Industrials""",0.05
"""WSM""","""Consumer Discretionary""",0.05
"""META""","""Communication Services""",0.05


In [8]:
positions.group_by("sector").agg(pl.col("weight").sum()).sort("weight", descending=True)

sector,weight
str,f64
"""Information Technology""",0.25
"""Financials""",0.2
"""Industrials""",0.2
"""Utilities""",0.1
"""Communication Services""",0.1
"""Consumer Discretionary""",0.1
"""Energy""",0.05


### Notes

This LP is well-conditioned for a naive dense simplex — 0/1 constraint coefficients, small integer/decimal RHS values, no huge scale mismatches — so it solves fast with no numerical drama. Working on this problem is what surfaced three real gaps in `optix.expressions` that are now fixed in `src/optix`, not papered over here:

- `Var` didn't support `<=`/`>=` directly (only `Expression` did) — broke every single-variable bound constraint (`weight[t] <= PER_ASSET_CAP`).
- `Expression` had `__add__` but no `__radd__` — broke `sum(...)` over expressions, since `sum` starts from `0 + first_item`.
- The ratio-test division in `pivot` throws a `RuntimeWarning: divide by zero` on every row where the pivot column is 0 — extremely common with sparse 0/1 constraints like these. Silenced since it's already correctly masked, not a real bug.

Natural next steps for this same dataset: a Markowitz mean-variance version once there's a QP solver (`load_returns(...).cov()` is right there), or a `>=`/`=` version (minimum sector exposure, cash-flow-style constraints) once there's a two-phase simplex.